# 01 · Encode the corpusTurns 84,942 issue texts into 384-dim vectors. **~2 minutes on a T4.**Runs on Colab because the development laptop has no GPU. The database stayslocal: `embed.py --export-texts` produces the input here, and`embed.py --import-vectors` loads the output back.### Upload before running| file | produced by ||---|---|| `texts_clean.jsonl.gz` | `python -m src.embed --export-texts colab/texts_clean.jsonl.gz --clean` |

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import torch, gcgc.collect(); torch.cuda.empty_cache()assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"print(torch.cuda.get_device_name(0),      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

### EncodeTwo settings are load-bearing:- **`normalize_embeddings=True`** — makes cosine similarity a plain dot product,  so retrieval is one matmul with no per-query norm.- **No bge query prefix.** bge's model card suggests  `"Represent this sentence for searching relevant passages:"`, but that is for  *asymmetric* retrieval — a short query against long passages. Here both sides  are issue text, so the task is symmetric and the prefix would put queries and  documents in different regions of the space.Output is written as numbered shards so a reclaimed Colab runtime costs oneshard rather than the whole run.

In [ ]:
import gzip, json, numpy as npfrom pathlib import Pathfrom sentence_transformers import SentenceTransformerMODEL = "BAAI/bge-small-en-v1.5"SHARD = 10_000recs = [json.loads(l) for l in gzip.open("texts_clean.jsonl.gz", "rt")]print(f"{len(recs):,} texts")model = SentenceTransformer(MODEL)Path("shards").mkdir(exist_ok=True)for s in range(0, len(recs), SHARD):    chunk = recs[s:s + SHARD]    vecs = model.encode([r["t"] for r in chunk], batch_size=64,                        normalize_embeddings=True, show_progress_bar=True,                        convert_to_numpy=True).astype(np.float32)    i = s // SHARD    np.save(f"shards/emb_{i:05d}.npy", vecs)    np.save(f"shards/ids_{i:05d}.npy", np.array([r["n"] for r in chunk], dtype=np.int64))    print("shard", i)

In [ ]:
!zip -qr shards.zip shardsfrom google.colab import filesfiles.download("shards.zip")

### Back on the laptop```bashunzip shards.zippython -m src.embed --import-vectors shards/ --clean```Import takes ~18 seconds. It was 1h22m until the MariaDB `VECTOR INDEX` wasdropped — HNSW insert cost grows with the graph already built, and nothing inthis project queries through that index (brute-force numpy is 32ms and exact;HNSW was 86ms and approximate). See NOTES.md, stage 3.